In [4]:
"""
TrainModel.py - DecepTech Machine Learning Algorithm
Multimodal Deception Detection System

This script processes audio and visual features from video segments
to classify truthful vs deceptive speech using the Bag of Lies dataset.
Now updated to use Wave2Vec2 for audio feature extraction.
"""
#!pip install transformers torchaudio


import pandas as pd
import numpy as np
import os
import pickle
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

class DecepTechML:
    """
    Multimodal Deception Detection Machine Learning Pipeline using Wave2Vec2
    """

    def __init__(self,
                 csv_path=r'C:\Users\barrettm5\git\DecepTech-2\csv\utterances.csv',
                 audio_dir=r'C:\Users\barrettm5\git\DecepTech-2\audio_clips\\'):
        self.csv_path = csv_path
        self.audio_dir = audio_dir
        self.label_encoder = LabelEncoder()
        self.model = None
        self.processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
        self.wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

    def load_data(self):
        try:
            self.df = pd.read_csv(self.csv_path)
            print(f"Loaded {len(self.df)} utterances from {self.csv_path}")
            return True
        except FileNotFoundError:
            print(f"Error: {self.csv_path} not found. Please verify the file path.")
            return False

    def extract_wave2vec_features(self, audio_path):
        try:
            speech, sr = torchaudio.load(audio_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                speech = resampler(speech)

            inputs = self.processor(speech.squeeze(), sampling_rate=16000, return_tensors="pt")
            with torch.no_grad():
                outputs = self.wav2vec_model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            return np.zeros(768)  # fallback vector

    def prepare_features(self):
        print("Extracting audio features using Wave2Vec2...")
        features = []
        labels = []

        for _, row in self.df.iterrows():
            audio_filename = f"{row['video_file'].replace('.mp4','')}_{int(row['start_time_ms'])}_{int(row['end_time_ms'])}.wav"
            audio_path = os.path.join(self.audio_dir, audio_filename)
            if not os.path.exists(audio_path):
                print(f"Missing audio: {audio_path}")
                continue
            feature_vec = self.extract_wave2vec_features(audio_path)
            features.append(feature_vec)
            labels.append(1 if row['veracity'].strip().lower() == 'truthful' else 0)


        self.features = np.array(features)
        self.labels = np.array(labels)
        print(f"Feature shape: {self.features.shape}, Labels shape: {self.labels.shape}")
        return self.features, self.labels

    def train_model(self):
        X_train, X_test, y_train, y_test = train_test_split(
            self.features, self.labels, test_size=0.2, random_state=42, stratify=self.labels)

        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.model.fit(X_train, y_train)

        y_pred = self.model.predict(X_test)
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        train_acc = accuracy_score(y_train, self.model.predict(X_train))
        print(f"Training Accuracy: {train_acc:.4f}")
        return self.model

    def save_model(self, output_path='deceptech_model.pkl'):
        with open(output_path, 'wb') as f:
            pickle.dump(self.model, f)
        print(f"Model saved to {output_path}")

# Run in Jupyter or script
print("Starting DecepTech ML Training...")
deceptech = DecepTechML()
if deceptech.load_data():
    features, labels = deceptech.prepare_features()
    model = deceptech.train_model()
    deceptech.save_model()
    print("\nDecepTech ML training complete! You can now use this model for predictions.")


Defaulting to user installation because normal site-packages is not writeable
Starting DecepTech ML Training...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded 225 utterances from C:\Users\barrettm5\git\DecepTech-2\csv\utterances.csv
Extracting audio features using Wave2Vec2...
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_41325_67960.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_67960_95050.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_95050_120037.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_146897_210761.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_232297_257500.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_257500_259270.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_259270_271620.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_271620_274755.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_274755_293520.wav
Missing audio: C:\Users\barrettm5\git\DecepTech-2\audio_clips\\23_BoL_293520_297280.wav